In [1]:
##extractor.py content
from pypdf import PdfReader

In [2]:
##the functions begin
##Extract text from notes PDFs, keeping track of which page each piece of text came from
def pdf_pages(pdf_path:str):
    reader = PdfReader(pdf_path)
    return [(i+1, page.extract_text()) for i, page in enumerate(reader.pages)]

In [4]:
##chunking up the pdf (it remembers its page number)
def chunk_pages(pages, chunk_size=400):
    chunks = []
    for page_num, text in pages:##this is the list of tuples created by our pdf_pages function
        ##check for blank pages (skip if stumbled upon) (pages with images and no text layer get skipped)
        if not text:
            continue
        words = text.split()##list of words in the text of that page number
        for i in range(0, len(words), chunk_size):
            chunks.append({"page": page_num, "text": " ".join(words[i:i+chunk_size])})
    return chunks

In [5]:
##some experimentation
pages = pdf_pages("/Users/tanishqsingh/Desktop/sample_notes.pdf")
chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks in {len(pages)} pages")

3 chunks in 3 pages


In [6]:
pages[:3]

[(1,
  'Operating Systems - Unit 1: Processes\nA process is a program in execution.\nEvery process has a unique Process ID (PID) assigned by the OS.\nProcess states: New, Ready, Running, Waiting, Terminated.\nA context switch saves the state of one process and loads another.\nThe PCB (Process Control Block) stores the process state, program\ncounter, CPU registers, and scheduling information.\nContext switching is pure overhead - no useful work is done during it.\n'),
 (2,
  'CPU Scheduling Algorithms\nFCFS (First Come First Served) is non-preemptive and can cause the\nconvoy effect, where short jobs wait behind a long one.\nSJF (Shortest Job First) gives the minimum average waiting time but\nrequires knowing burst times in advance, which is not practical.\nRound Robin uses a fixed time quantum, typically 10-100 milliseconds.\nIf the quantum is too large, Round Robin degrades into FCFS.\nPriority scheduling can cause starvation, solved by aging.\n'),
 (3,
  'Memory Management\nPaging d

In [7]:
chunks[0]

{'page': 1,
 'text': 'Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.'}

In [11]:
##embeding the chunks that we made in extractor.py
import os
from dotenv import load_dotenv
from google import genai
import chromadb

load_dotenv()##loading the environment vars in the OS

client = genai.Client()
EMBED_M = "gemini-embedding-2"

def embed_text(texts):
    vectors = []
    for text in texts:
        result = client.models.embed_content(
            model=EMBED_M,
            contents=text)
        vectors.append(result.embeddings[0].values)
    return vectors

def build_store(chunks):
    chroma_client = chromadb.PersistentClient()
    notes = chroma_client.get_or_create_collection(name="my_notes")
    vector = embed_text([c["text"] for c in chunks])
    notes.add(
                    ids=[f"c-{i}" for i in range(0,len(chunks))],
                    embeddings=vector,
                    documents=[c["text"] for c in chunks],
                    metadatas=[{"page":c["page"]} for c in chunks],) ##meta data is always list of dicts
    #print(notes.count()) just for checking purposes
    return notes

In [ ]:
##some testing
build_store(chunks)

In [16]:
##some testing (kill me)
pages = pdf_pages("/Users/tanishqsingh/Desktop/sample_notes.pdf")
chunks = chunk_pages(pages)
coll = build_store(chunks)

In [50]:
##now working on the retrieval file content which has functions for retrival of document from chroma db
def doc_retrieval(ip_query, notes, n_result=3):
    re_doc = []
    vectors = embed_text([ip_query])##using our vector function
    result = notes.query(
                query_embeddings=vectors,
                n_results=n_result
    )
    for i,j in zip(result['metadatas'][0],result['documents'][0]):
        re_doc.append({'page':i['page'], 'text':j})
    return re_doc

In [17]:
matching = doc_retrieval("what is the convoy effect?",coll)
print(matching)

{'ids': [['c-1', 'c-0', 'c-2']], 'embeddings': None, 'documents': [['CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.', 'Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.', '

In [43]:
##some more testing= kill meeeeeeeeee
re_doc = []
for i,j in zip(matching['metadatas'][0],matching['documents'][0]):
    print(i,j)

{'page': 2} CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.
{'page': 1} Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.
{'page': 3} Memory Management Paging divides memory

In [39]:
a = ("John", "Charles", "Mike")
b = ["Jenny", "Christy", "Monica"]
x = zip(a, b)
for i in x:
    print(i)

('John', 'Jenny')
('Charles', 'Christy')
('Mike', 'Monica')


In [51]:
##even more testing
doc_retrieval("what is the convoy effect?", coll, n_result=1)

[{'page': 2,
  'text': 'CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.'}]